In [1]:
import asyncio

from agent_framework import WorkflowBuilder, WorkflowContext, executor, AgentResponseUpdate
from typing_extensions import Never
from utils import new_base_openai_agent, new_base_gemini_agent

In [17]:
@executor(id="upper_case")
async def upper_case(text: str, ctx: WorkflowContext[str, str]):
    await ctx.send_message(text.upper())
    await ctx.yield_output('Upper Case is done')

@executor(id="reverse_text")
async def reverse_text(text: str, ctx: WorkflowContext[Never, str]) -> None:
    await asyncio.sleep(3)
    await ctx.yield_output(text[::-1])

workflow = (
    WorkflowBuilder(start_executor=upper_case)
    .add_edge(upper_case, reverse_text)
    .build()
)

## Non-Streaming Execution

In [15]:
result = await workflow.run("hello world")
print(result.get_outputs())               # list of terminal outputs
print(result.get_intermediate_outputs())  # list of intermediate outputs
print(result.get_final_state())           # WorkflowRunState.IDLE

['Upper Case is done', 'DLROW OLLEH']
[]
WorkflowRunState.IDLE


## Streaming Execution

In [10]:
async for event in workflow.run("hello world", stream=True):
    print(event)

WorkflowEvent(type='started')
WorkflowEvent(type='status', state=IN_PROGRESS)
WorkflowEvent(type='superstep_started', iteration=1)
WorkflowEvent(type='executor_invoked', executor_id='WriterAgent', data='hello world')
WorkflowEvent(type='output', executor_id='WriterAgent', data=<agent_framework._types.AgentResponseUpdate object at 0x000001E798E07650>)
WorkflowEvent(type='output', executor_id='WriterAgent', data=<agent_framework._types.AgentResponseUpdate object at 0x000001E798E07410>)
WorkflowEvent(type='output', executor_id='WriterAgent', data=<agent_framework._types.AgentResponseUpdate object at 0x000001E798E077D0>)
WorkflowEvent(type='output', executor_id='WriterAgent', data=<agent_framework._types.AgentResponseUpdate object at 0x000001E798E07D10>)
WorkflowEvent(type='output', executor_id='WriterAgent', data=<agent_framework._types.AgentResponseUpdate object at 0x000001E798E07E90>)
WorkflowEvent(type='output', executor_id='WriterAgent', data=<agent_framework._types.AgentResponseUpdat

**What are supersteps**

Important: Within a single superstep, all triggered executors run in parallel, but the workflow does not advance to the next superstep until every executor completes. This affects fan-out patterns: if you fan out to multiple paths — one with a chain of executors and another with a single long-running executor — the chained path cannot advance until the long-running executor completes.

Workflow execution is organized into discrete supersteps. Each superstep:

- Collects all pending messages from the previous superstep
- Routes messages to target executors based on edge definitions
- Runs all target executors concurrently within the superstep
- Waits for all executors to complete before advancing (synchronization barrier)
- Queues any new messages emitted by executors for the next superstep

**Why supersteps**

The BSP model provides important guarantees:

- Deterministic execution: Given the same input, the workflow always executes in the same order
- Reliable checkpointing: State can be saved at superstep boundaries for fault tolerance
- Simpler reasoning: No race conditions between supersteps; each sees a consistent view of messages

If you need truly independent parallel paths that don't block each other, consolidate sequential steps into a single executor. Instead of chaining `step1 → step2 → step3`, combine that logic into one executor. Both parallel paths then execute within a single superstep.

 Pregel execution model: https://kowshik.github.io/JPregel/pregel_paper.pdf

## Agents in Workflows

In [11]:
writer_agent = new_base_openai_agent(
    name="WriterAgent",
    description="An agent that assists with writing tasks.",
    instructions=(
        "You are an excellent content writer."
        "You create new content and edit contents based on the feedback."
    )
)

reviewer_agent = new_base_openai_agent(
    name="Reviewer",
    description="An agent that reviews and provides feedback on content.",
    instructions=(
        "You are an excellent content reviewer. "
        "Provide actionable feedback to the writer about the provided content. "
        "Provide the feedback in the most concise manner possible."
    )
)

# reviewer_agent = new_base_gemini_agent(
#     name="Reviewer",
#     description="An agent that reviews and provides feedback on content.",
#     instructions=(
#         "You are an excellent content reviewer. "
#         "Provide actionable feedback to the writer about the provided content. "
#         "Provide the feedback in the most concise manner possible."
#     )
# )

In [3]:
workflow = (
    WorkflowBuilder(start_executor=writer_agent)
        .add_edge(writer_agent, reviewer_agent)
        .build()
)

In [5]:
events = workflow.run("Create a slogan for a new electric SUV that is affordable and fun to drive.", stream=True)
agent_responses: list[AgentResponseUpdate] = [] 
async for event in events:
    print(event)
    if event.type == 'output' and isinstance(event.data, AgentResponseUpdate):
        agent_responses.append(event.data)

WorkflowEvent(type='started')
WorkflowEvent(type='status', state=IN_PROGRESS)
WorkflowEvent(type='superstep_started', iteration=1)
WorkflowEvent(type='executor_invoked', executor_id='WriterAgent', data='Create a slogan for a new electric SUV that is affordable and fun to drive.')
WorkflowEvent(type='output', executor_id='WriterAgent', data=<agent_framework._types.AgentResponseUpdate object at 0x000001E79779AED0>)
WorkflowEvent(type='output', executor_id='WriterAgent', data=<agent_framework._types.AgentResponseUpdate object at 0x000001E79779A450>)
WorkflowEvent(type='output', executor_id='WriterAgent', data=<agent_framework._types.AgentResponseUpdate object at 0x000001E79779B110>)
WorkflowEvent(type='output', executor_id='WriterAgent', data=<agent_framework._types.AgentResponseUpdate object at 0x000001E797ABC350>)
WorkflowEvent(type='output', executor_id='WriterAgent', data=<agent_framework._types.AgentResponseUpdate object at 0x000001E797ABC950>)
WorkflowEvent(type='output', executor_i

In [6]:
last_author: str | None = None
for update in agent_responses:
    author = update.author_name
    if author != last_author:
        if last_author is not None:
            print()
        print(f"{author}: {update.text}", end="", flush=True)
        last_author = author
    else:
        print(update.text, end="", flush=True)

WriterAgent: “Electric thrills, family bills—finally both in one SUV.”“Electric thrills, family bills—finally both in one SUV.”
Reviewer: - “Electric thrills, family bills—finally both in one SUV.” is witty, but “bills” evokes paying expenses, which can feel negative and may dilute the “affordable” message.  
- The structure is slightly complex; slogans usually benefit from simpler, punchier phrasing.  
- Try a more straightforward value + fun combo, e.g.:  
  - “All the fun. Half the cost. One electric SUV.”  
  - “Affordable to own, unforgettable to drive—your electric SUV.”  
- Make sure “electric SUV” appears clearly and the affordability angle feels positive (savings, value) rather than about bills or costs.- “Electric thrills, family bills—finally both in one SUV.” is witty, but “bills” evokes paying expenses, which can feel negative and may dilute the “affordable” message.  
- The structure is slightly complex; slogans usually benefit from simpler, punchier phrasing.  
- Try a m

## Agents as Workflows

By wrapping your workflow as an Agent, you can interact with it through the same familiar API you'd use for a simple chat agent.

When you convert a workflow to an agent:

- The workflow is validated to ensure its start executor can accept the required input types
- A session is created to manage conversation state
- Input messages are routed to the workflow's start executor
- Workflow events are converted to agent response updates
- External input requests (from RequestInfoExecutor) are surfaced as function calls

In [12]:
workflow = (
    WorkflowBuilder(start_executor=writer_agent)
        .add_edge(writer_agent, reviewer_agent)
        .build()
)

workflow_agent = workflow.as_agent(name="Content Pipeline Agent")

In [15]:
events = workflow_agent.run("Create a slogan for a new electric SUV that is affordable and fun to drive.", stream=True)
async for event in events:
    print(event.to_json(indent=2))

{
  "type": "agent_response_update",
  "contents": [],
  "role": "assistant",
  "author_name": "WriterAgent",
  "response_id": "resp_0dde638642482b62006adf37a66674819088fb8d00e4fb148a",
  "continuation_token": {
    "response_id": "resp_0dde638642482b62006adf37a66674819088fb8d00e4fb148a"
  },
  "additional_properties": {},
  "_operation_state": "in_progress"
}
{
  "type": "agent_response_update",
  "contents": [],
  "role": "assistant",
  "author_name": "WriterAgent",
  "response_id": "resp_0dde638642482b62006adf37a66674819088fb8d00e4fb148a",
  "continuation_token": {
    "response_id": "resp_0dde638642482b62006adf37a66674819088fb8d00e4fb148a"
  },
  "additional_properties": {},
  "_operation_state": "in_progress"
}
{
  "type": "agent_response_update",
  "contents": [],
  "role": "assistant",
  "author_name": "WriterAgent",
  "additional_properties": {},
  "_operation_state": "in_progress"
}
{
  "type": "agent_response_update",
  "contents": [
    {
      "type": "text",
      "text": 

**Note:** "contents" has a merge_mode of either "append" or "replace". Do normal agents have this, too?

Yes, they do; however, "replace" only shows-up for "usage" events. Workflows must be able to issue it for their text because of how the **superstep** is created.

## How to pause/stop a running workflow